In [2]:
import os
import zipfile
import requests
import numpy as np
import pandas as pd
import plotly.express as px
import folium

from tqdm import tqdm
from pathlib import Path

In [3]:
trip_demo = pd.DataFrame({
    "ride_id": list(range(1, 21)),

    "start_station": [
        "Station A", "Station A", "Station A", "Station A", "Station A",
        "Station B", "Station B", "Station B", "Station B",
        "Station C", "Station C", "Station C",
        "Station D", "Station D", "Station D",
        "Station E", "Station E",
        "Station A", "Station B", "Station C"
    ],

    "end_station": [
        "Station B", "Station B", "Station B", "Station C", "Station C",
        "Station A", "Station A", "Station C", "Station D",
        "Station A", "Station B", "Station E",
        "Station A", "Station C", "Station E",
        "Station A", "Station D",
        "Station E", "Station E", "Station D"
    ],

    "started_at": pd.to_datetime([
        "2025-01-01 08:00", "2025-01-01 08:15", "2025-01-01 08:30",
        "2025-01-01 09:00", "2025-01-01 09:20",
        "2025-01-01 10:00", "2025-01-01 10:15", "2025-01-01 10:40",
        "2025-01-01 11:00",
        "2025-01-01 11:30", "2025-01-01 12:00", "2025-01-01 12:20",
        "2025-01-01 13:00", "2025-01-01 13:30", "2025-01-01 14:00",
        "2025-01-01 14:30", "2025-01-01 15:00",
        "2025-01-01 15:30", "2025-01-01 16:00", "2025-01-01 16:30"
    ]),

    "ended_at": pd.to_datetime([
        "2025-01-01 08:10", "2025-01-01 08:28", "2025-01-01 08:42",
        "2025-01-01 09:18", "2025-01-01 09:35",
        "2025-01-01 10:12", "2025-01-01 10:30", "2025-01-01 10:55",
        "2025-01-01 11:18",
        "2025-01-01 11:48", "2025-01-01 12:14", "2025-01-01 12:45",
        "2025-01-01 13:20", "2025-01-01 13:48", "2025-01-01 14:20",
        "2025-01-01 14:55", "2025-01-01 15:22",
        "2025-01-01 15:58", "2025-01-01 16:25", "2025-01-01 16:50"
    ]),

    "member_casual": [
        "member", "member", "casual", "member", "casual",
        "member", "member", "casual", "member",
        "casual", "member", "casual",
        "member", "casual", "member",
        "casual", "member",
        "member", "casual", "member"
    ]
})

trip_demo

,ride_id,start_station,end_station,started_at,ended_at,member_casual
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual
5,6,Station B,Station A,2025-01-01 10:00:00,2025-01-01 10:12:00,member
6,7,Station B,Station A,2025-01-01 10:15:00,2025-01-01 10:30:00,member
7,8,Station B,Station C,2025-01-01 10:40:00,2025-01-01 10:55:00,casual
8,9,Station B,Station D,2025-01-01 11:00:00,2025-01-01 11:18:00,member
9,10,Station C,Station A,2025-01-01 11:30:00,2025-01-01 11:48:00,casual


In [4]:
station_coordinates = pd.DataFrame({
    "station": [
        "Station A",
        "Station B",
        "Station C",
        "Station D",
        "Station E"
    ],
    "lat": [
        40.735,
        40.751,
        40.742,
        40.728,
        40.760
    ],
    "lng": [
        -73.991,
        -73.977,
        -73.985,
        -73.970,
        -73.995
    ]
})

station_coordinates

,station,lat,lng
0,Station A,40.735,-73.991
1,Station B,40.751,-73.977
2,Station C,40.742,-73.985
3,Station D,40.728,-73.970
4,Station E,40.760,-73.995


In [5]:
trip_demo = (
    trip_demo
    .merge(
        station_coordinates,
        left_on="start_station",
        right_on="station",
        how="left"
    )
    .rename(columns={
        "lat": "start_lat",
        "lng": "start_lng"
    })
    .drop(columns="station")
    .merge(
        station_coordinates,
        left_on="end_station",
        right_on="station",
        how="left"
    )
    .rename(columns={
        "lat": "end_lat",
        "lng": "end_lng"
    })
    .drop(columns="station")
)

trip_demo.head()

,ride_id,start_station,end_station,started_at,ended_at,member_casual,start_lat,start_lng,end_lat,end_lng
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member,40.735,-73.991,40.751,-73.977
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member,40.735,-73.991,40.751,-73.977
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual,40.735,-73.991,40.751,-73.977
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member,40.735,-73.991,40.742,-73.985
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual,40.735,-73.991,40.742,-73.985


In [6]:
start_df = ['ride_id', 'start_station','started_at','start_lat', 'start_lng']
end_df = ['ride_id', 'ended_station','end_at','end_lat', 'end_lng']

## Map center

In [7]:
map_center = [
    pd.concat([trip_demo["start_lat"], trip_demo["end_lat"]]).mean(),
    pd.concat([trip_demo["start_lng"], trip_demo["end_lng"]]).mean()
]

map_center

[np.float64(40.7427), np.float64(-73.9841)]

## Adding Duration

In [8]:
trip_demo["duration_min"] = (
    trip_demo["ended_at"] - trip_demo["started_at"]
).dt.total_seconds() / 60

trip_demo[[
    "ride_id",
    "start_station",
    "end_station",
    "started_at",
    "ended_at",
    "duration_min"
]].head()

,ride_id,start_station,end_station,started_at,ended_at,duration_min
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,10.0
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,13.0
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,12.0
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,18.0
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,15.0


## Folium

In [9]:
m = folium.Map(
    location=[40.74, -73.99],
    zoom_start=13
)

for _, row in station_coordinates.iterrows():
    folium.Marker(
        location=[row["lat"], row["lng"]],
        popup=row["station"]
    ).add_to(m)

m

## LIne

In [10]:
m = folium.Map(
    location=[40.74, -73.99],
    zoom_start=13
)

start_point = [trip_demo.loc[0, "start_lat"], trip_demo.loc[0, "start_lng"]]
end_point = [trip_demo.loc[0, "end_lat"], trip_demo.loc[0, "end_lng"]]

folium.Marker(start_point, popup="Start").add_to(m)
folium.Marker(end_point, popup="End").add_to(m)

folium.PolyLine(
    locations=[start_point, end_point],
    weight=5,
    opacity=0.8
    
).add_to(m)

m

## Flow Data

In [11]:
flow_data = (
    trip_demo
    .groupby(
        ["start_station", "end_station"],
        as_index=False
    )
    .agg(
        number_of_rides=("ride_id", "count"),
        avg_duration_min=("duration_min", "mean"),
        start_lat=("start_lat", "mean"),
        start_lng=("start_lng", "mean"),
        end_lat=("end_lat", "mean"),
        end_lng=("end_lng", "mean")
    )
    .sort_values("number_of_rides", ascending=False)
)

flow_data

,start_station,end_station,number_of_rides,avg_duration_min,start_lat,start_lng,end_lat,end_lng
0,Station A,Station B,3,11.666667,40.735,-73.991,40.751,-73.977
1,Station A,Station C,2,16.500000,40.735,-73.991,40.742,-73.985
3,Station B,Station A,2,13.500000,40.751,-73.977,40.735,-73.991
2,Station A,Station E,1,28.000000,40.735,-73.991,40.760,-73.995
4,Station B,Station C,1,15.000000,40.751,-73.977,40.742,-73.985
5,Station B,Station D,1,18.000000,40.751,-73.977,40.728,-73.970
6,Station B,Station E,1,25.000000,40.751,-73.977,40.760,-73.995
7,Station C,Station A,1,18.000000,40.742,-73.985,40.735,-73.991
8,Station C,Station B,1,14.000000,40.742,-73.985,40.751,-73.977
9,Station C,Station D,1,20.000000,40.742,-73.985,40.728,-73.970


## Bounding Box

In [12]:
min_lat = min(trip_demo["start_lat"].min(), trip_demo["end_lat"].min())
max_lat = max(trip_demo["start_lat"].max(), trip_demo["end_lat"].max())

min_lng = min(trip_demo["start_lng"].min(), trip_demo["end_lng"].min())
max_lng = max(trip_demo["start_lng"].max(), trip_demo["end_lng"].max())

bounding_box = pd.DataFrame({
    "metric": ["min_lat", "max_lat", "min_lng", "max_lng"],
    "value": [min_lat, max_lat, min_lng, max_lng]
})

bounding_box

,metric,value
0,min_lat,40.728
1,max_lat,40.760
2,min_lng,-73.995
3,max_lng,-73.970


In [13]:
bbox_map = folium.Map(
    location=map_center,
    zoom_start=13,
    tiles="CartoDB positron"
)

for _, row in station_coordinates.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lng"]],
        radius=6,
        popup=row["station"],
        tooltip=row["station"],
        fill=True
    ).add_to(bbox_map)

bbox_coordinates = [
    [min_lat, min_lng],
    [min_lat, max_lng],
    [max_lat, max_lng],
    [max_lat, min_lng],
    [min_lat, min_lng]
]

folium.PolyLine(
    locations=bbox_coordinates,
    weight=4,
    opacity=0.8,
    popup="Bounding Box"
).add_to(bbox_map)

bbox_map

## Visualizing Flows

In [14]:
flow_map = folium.Map(
    location=map_center,
    zoom_start=13,
    tiles="CartoDB positron"
)

for _, row in station_coordinates.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lng"]],
        radius=6,
        popup=row["station"],
        tooltip=row["station"],
        fill=True
    ).add_to(flow_map)

max_rides = flow_data["number_of_rides"].max()

for _, row in flow_data.iterrows():

    start_point = [row["start_lat"], row["start_lng"]]
    end_point = [row["end_lat"], row["end_lng"]]

    line_width = 1 + 7 * row["number_of_rides"] / max_rides

    popup_text = (
        f"<b>{row['start_station']} → {row['end_station']}</b><br>"
        f"Number of rides: {row['number_of_rides']}<br>"
        f"Average duration: {row['avg_duration_min']:.1f} minutes"
    )

    folium.PolyLine(
        locations=[start_point, end_point],
        color='red',
        weight=line_width,
        opacity=0.7,
        popup=popup_text,
        tooltip=f"{row['start_station']} → {row['end_station']}"
    ).add_to(flow_map)

flow_map

## Distance

In [15]:
import geopandas as gpd
start_points = gpd.GeoDataFrame(
    trip_demo.copy(),
    geometry=gpd.points_from_xy(
        trip_demo["start_lng"],
        trip_demo["start_lat"]
    ),
    crs="EPSG:4326"
)

end_points = gpd.GeoDataFrame(
    trip_demo.copy(),
    geometry=gpd.points_from_xy(
        trip_demo["end_lng"],
        trip_demo["end_lat"]
    ),
    crs="EPSG:4326"
)

In [16]:
start_points[[
    "ride_id",
    "start_station",
    "end_station",
    "geometry"
]].head()

,ride_id,start_station,end_station,geometry
0,1,Station A,Station B,POINT (-73.991 40.735)
1,2,Station A,Station B,POINT (-73.991 40.735)
2,3,Station A,Station B,POINT (-73.991 40.735)
3,4,Station A,Station C,POINT (-73.991 40.735)
4,5,Station A,Station C,POINT (-73.991 40.735)


In [17]:
projected_crs = "EPSG:32618"

start_points_projected = start_points.to_crs(projected_crs)
end_points_projected = end_points.to_crs(projected_crs)

In [18]:
trip_demo["distance_m"] = start_points_projected.geometry.distance(
    end_points_projected.geometry
)

trip_demo["distance_km"] = trip_demo["distance_m"] / 1000

trip_demo[[
    "ride_id",
    "start_station",
    "end_station",
    "duration_min",
    "distance_m",
    "distance_km"
]].head()

,ride_id,start_station,end_station,duration_min,distance_m,distance_km
0,1,Station A,Station B,10.0,2133.621698,2.133622
1,2,Station A,Station B,13.0,2133.621698,2.133622
2,3,Station A,Station B,12.0,2133.621698,2.133622
3,4,Station A,Station C,18.0,927.671157,0.927671
4,5,Station A,Station C,15.0,927.671157,0.927671


## Distance Summary

In [19]:
trip_demo[["distance_m", "distance_km"]].describe()

,distance_m,distance_km
count,20.000000,20.000000
mean,2113.717746,2.113718
std,898.892579,0.898893
min,927.671157,0.927671
25%,1665.451089,1.665451
50%,2133.621698,2.133622
75%,2282.181051,2.282181
max,4132.274455,4.132274


In [20]:
route_summary = (
    trip_demo
    .groupby(
        ["start_station", "end_station"],
        as_index=False
    )
    .agg(
        number_of_rides=("ride_id", "count"),
        avg_duration_min=("duration_min", "mean"),
        median_duration_min=("duration_min", "median"),
        avg_distance_km=("distance_km", "mean"),
        median_distance_km=("distance_km", "median"),
        start_lat=("start_lat", "mean"),
        start_lng=("start_lng", "mean"),
        end_lat=("end_lat", "mean"),
        end_lng=("end_lng", "mean")
    )
    .sort_values("number_of_rides", ascending=False)
)

route_summary

,start_station,end_station,number_of_rides,avg_duration_min,median_duration_min,avg_distance_km,median_distance_km,start_lat,start_lng,end_lat,end_lng
0,Station A,Station B,3,11.666667,12.0,2.133622,2.133622,40.735,-73.991,40.751,-73.977
1,Station A,Station C,2,16.500000,16.5,0.927671,0.927671,40.735,-73.991,40.742,-73.985
3,Station B,Station A,2,13.500000,13.5,2.133622,2.133622,40.751,-73.977,40.735,-73.991
2,Station A,Station E,1,28.000000,28.0,2.795834,2.795834,40.735,-73.991,40.760,-73.995
4,Station B,Station C,1,15.000000,15.0,1.206023,1.206023,40.751,-73.977,40.742,-73.985
5,Station B,Station D,1,18.000000,18.0,2.620861,2.620861,40.751,-73.977,40.728,-73.970
6,Station B,Station E,1,25.000000,25.0,1.818594,1.818594,40.751,-73.977,40.760,-73.995
7,Station C,Station A,1,18.000000,18.0,0.927671,0.927671,40.742,-73.985,40.735,-73.991
8,Station C,Station B,1,14.000000,14.0,1.206023,1.206023,40.742,-73.985,40.751,-73.977
9,Station C,Station D,1,20.000000,20.0,2.005000,2.005000,40.742,-73.985,40.728,-73.970


In [21]:
start_points.geometry.distance(end_points.geometry)

C:\Users\Sargsyan\AppData\Local\Temp\ipykernel_17964\340217584.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  start_points.geometry.distance(end_points.geometry)


0     0.021260
1     0.021260
2     0.021260
3     0.009220
4     0.009220
5     0.021260
6     0.021260
7     0.012042
8     0.024042
9     0.009220
10    0.012042
11    0.020591
12    0.022136
13    0.020518
14    0.040608
15    0.025318
16    0.040608
17    0.025318
18    0.020125
19    0.020518
dtype: float64

## Citi Bike Jersey

### Step 1: Define the Data Source

In [22]:
CITIBIKE_INDEX_URL = "https://s3.amazonaws.com/tripdata"
OUTPUT_DIR = "../data/citibike"
PERIOD = "202510"

In [23]:
file_name = f"JC-{PERIOD}-citibike-tripdata.zip"
url = f"{CITIBIKE_INDEX_URL}/{file_name}"
url

'https://s3.amazonaws.com/tripdata/JC-202510-citibike-tripdata.zip'

### Step 2: Download Jersey Data

In [24]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import requests

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

for i in PERIODS:
    urls = [
    f"{CITIBIKE_INDEX_URL}/JC-{i}-citibike-tripdata.csv.zip",
    f"{CITIBIKE_INDEX_URL}/JC-{i}-citibike-tripdata.zip",
]

    last_error = None

    for url in urls:
        try:
            print(f"Trying: {url}")

            response = requests.get(url, timeout=30)
            response.raise_for_status()

            with ZipFile(BytesIO(response.content), "r") as zip_ref:
                zip_ref.extractall(output_dir)

            print(f"Success: {url}")
            break

        except Exception as e:
            print(f"Failed: {url}")
            last_error = e

    else:
        raise RuntimeError(f"Both URLs failed for period {i}") from last_error

NameError: name 'PERIODS' is not defined

## Step 3: Read the Data

In [ ]:
file_name  = 'JC-202510-citibike-tripdata.csv'
citibike_202510 = pd.read_csv(f'{output_dir}/{file_name}')
citibike_202510.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,929852F0DC04F49C,electric_bike,2025-10-08 06:59:53.767,2025-10-08 07:04:08.012,Warren St,JC006,Essex Light Rail,NaN,40.721124,-74.038051,NaN,NaN,member
1,7E6F8FADF45B1D8E,electric_bike,2025-10-08 08:13:15.096,2025-10-08 08:18:16.168,Newark Ave,JC032,Essex Light Rail,NaN,40.721525,-74.046305,NaN,NaN,member
2,9B679869FCA8F845,electric_bike,2025-10-09 10:10:26.020,2025-10-09 10:20:08.505,Liberty Light Rail,JC052,Newport PATH,NaN,40.711242,-74.055701,NaN,NaN,member
3,E2051DD457BC4076,electric_bike,2025-10-09 11:34:22.937,2025-10-09 11:48:49.501,Hoboken Ave at Monmouth St,JC105,Newport PATH,NaN,40.735208,-74.046964,NaN,NaN,member
4,239AC07371E414EC,electric_bike,2025-10-23 19:24:16.237,2025-10-23 19:28:22.592,Union St & Bergen Ave,JC122,Garfield Light Rail,JC119,40.715750,-74.078870,40.710526,-74.070112,member


## Step 4: Downloading year 2025

In [ ]:
def period_iterator(year:list,start_m:int, stop_m:int)->list:
    """
    
    """
    YEAR = year
    MONTH =  [str(i+1) if i+1>9 else "0" + str(i+1) for i in range(start_m, stop_m)]

    periods = []

    for i in YEAR:
        for j in MONTH:
            k = i+j
            periods.append(k)
    # print(periods)
    return periods

In [ ]:
periods  = period_iterator(year = ['2025'],start_m = 1, stop_m = 12)

periods

['202502',
 '202503',
 '202504',
 '202505',
 '202506',
 '202507',
 '202508',
 '202509',
 '202510',
 '202511',
 '202512']

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

from urllib.error import HTTPError, URLError

CITIBIKE_INDEX_URL = "https://s3.amazonaws.com/tripdata"
OUTPUT_DIR = "../data/citibike"
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(exist_ok=True)


for i in PERIODS:

    try:
        file_name = f"JC-{i}-citibike-tripdata.csv.zip"
        url = f"{CITIBIKE_INDEX_URL}/{file_name}"

        zip_path = output_dir / file_name
        urlretrieve(url, zip_path)

    except (HTTPError, URLError, FileNotFoundError):
        file_name = f"JC-{i}-citibike-tripdata.zip"
        url = f"{CITIBIKE_INDEX_URL}/{file_name}"

        zip_path = output_dir / file_name
        urlretrieve(url, zip_path)

    with ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(output_dir)
    print(f'{file_name}  Extracted')
    zip_path.unlink()
    print(f"{file_name} removed.")


JC-202501-citibike-tripdata.csv.zip  Extracted
JC-202501-citibike-tripdata.csv.zip removed.
JC-202502-citibike-tripdata.csv.zip  Extracted
JC-202502-citibike-tripdata.csv.zip removed.
JC-202503-citibike-tripdata.csv.zip  Extracted
JC-202503-citibike-tripdata.csv.zip removed.
JC-202504-citibike-tripdata.csv.zip  Extracted
JC-202504-citibike-tripdata.csv.zip removed.
JC-202505-citibike-tripdata.csv.zip  Extracted
JC-202505-citibike-tripdata.csv.zip removed.
JC-202506-citibike-tripdata.csv.zip  Extracted
JC-202506-citibike-tripdata.csv.zip removed.
JC-202507-citibike-tripdata.csv.zip  Extracted
JC-202507-citibike-tripdata.csv.zip removed.
JC-202508-citibike-tripdata.csv.zip  Extracted
JC-202508-citibike-tripdata.csv.zip removed.
JC-202509-citibike-tripdata.csv.zip  Extracted
JC-202509-citibike-tripdata.csv.zip removed.
JC-202510-citibike-tripdata.zip  Extracted
JC-202510-citibike-tripdata.zip removed.
JC-202511-citibike-tripdata.csv.zip  Extracted
JC-202511-citibike-tripdata.csv.zip remov

## Step 5: Removing __MACOSX

In [ ]:
import shutil

shutil.rmtree(output_dir / "__MACOSX")

## Step 6: Concatinating

In [ ]:
import glob
import numpy as np
import pandas as pd

file_names = glob.glob(f'{output_dir}/*.csv')



dfs = []
cols = []
for file_name in file_names:
    df = pd.read_csv(file_name)
    print(df.columns, 2*"||",len(df.columns))

    cols.append(list(df.columns))
    dfs.append(df)

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str') |||| 13
Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str') |||| 13
Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str') |||| 13
Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      d

In [ ]:
citibike_df = pd.concat(dfs)
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,40.742677,-74.051789,member
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,40.711130,-74.078900,member
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member


In [ ]:
df.to_csv(f"{output_dir}/JC/JC2025.csv",index=False)

In [29]:
citibike_df=pd.read_csv(f"{output_dir}/JC/JC2025.csv",
                        parse_dates=['started_at','ended_at'])

## Step 7: Setting Dates

In [30]:
citibike_df['started_at'] = pd.to_datetime(citibike_df['started_at'],errors="coerce")
citibike_df['ended_at'] = pd.to_datetime(citibike_df['ended_at'],errors="coerce")

## Step 8: df overview

In [31]:
citibike_df.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str')

## Step 9: Missing Values

In [32]:
missing_values = (
    citibike_df
    .isna()
    .sum()
    .reset_index()
)

missing_values.columns = ["column", "missing_count"]

missing_values["missing_share"] = (
    missing_values["missing_count"] / len(citibike_df)
)

missing_values.sort_values("missing_count", ascending=False)

,column,missing_count,missing_share
7,end_station_id,606,0.012502
10,end_lat,606,0.012502
11,end_lng,606,0.012502
6,end_station_name,352,0.007262
0,ride_id,0,0.000000
4,start_station_name,0,0.000000
3,ended_at,0,0.000000
2,started_at,0,0.000000
1,rideable_type,0,0.000000
8,start_lat,0,0.000000


## Step 10: Ride Duration

In [33]:
citibike_df["ride_duration_minutes"] = (
    citibike_df["ended_at"] - citibike_df["started_at"]
).dt.total_seconds() / 60

In [34]:
citibike_df = citibike_df.dropna(
    subset=[
        "ride_id",
        "started_at",
        "ended_at",
        "start_lat",
        "start_lng",
        "end_lat",
        "end_lng"
    ]
)

citibike_df = citibike_df[
    (citibike_df["ride_duration_minutes"] > 1) &
    (citibike_df["ride_duration_minutes"] <= 24 * 60)
].copy()

## Step 11: Time Based Variables

In [35]:
citibike_df["date"] = citibike_df["started_at"].dt.date
citibike_df["month"] = citibike_df["started_at"].dt.to_period("M").astype(str)
citibike_df["month_name"] = citibike_df["started_at"].dt.month_name()
citibike_df["day_of_week"] = citibike_df["started_at"].dt.day_name()
citibike_df["hour"] = citibike_df["started_at"].dt.hour

In [36]:
def assign_season(month_number):
    if month_number in [12, 1, 2]:
        return "Winter"
    elif month_number in [3, 4, 5]:
        return "Spring"
    elif month_number in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


citibike_df["season"] = (
    citibike_df["started_at"]
    .dt.month
    .apply(assign_season)
)

In [37]:
citibike_df[
    [
        "started_at",
        "date",
        "month",
        "month_name",
        "day_of_week",
        "hour",
        "season"
    ]
].head()

,started_at,date,month,month_name,day_of_week,hour,season
0,2025-12-26 10:20:09.188,2025-12-26,2025-12,December,Friday,10,Winter
1,2025-12-16 06:48:28.111,2025-12-16,2025-12,December,Tuesday,6,Winter
2,2025-12-20 12:50:55.361,2025-12-20,2025-12,December,Saturday,12,Winter
3,2025-12-22 16:59:42.922,2025-12-22,2025-12,December,Monday,16,Winter
4,2025-12-17 18:06:09.352,2025-12-17,2025-12,December,Wednesday,18,Winter


In [38]:
citibike_df.to_csv("../data/citibike/JC/JC2025_Enriched.csv", index = False)

## Step 12: Getting Weather Data

In [39]:
import pandas as pd
import requests

lat = 40.7178
lng = -74.0431

start_date = "2025-01-01"
end_date = "2025-12-31"

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": lat,
    "longitude": lng,
    "start_date": start_date,
    "end_date": end_date,
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "temperature_2m_mean",
        "precipitation_sum",
        "rain_sum",
        "snowfall_sum",
        "wind_speed_10m_max"
    ],
    "timezone": "America/New_York"
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()

weather_daily = pd.DataFrame(data["daily"])
weather_daily["date"] = pd.to_datetime(weather_daily["time"])
weather_daily = weather_daily.drop(columns="time")

weather_daily.head()

,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max,date
0,10.9,3.9,7.4,4.5,4.5,0.0,23.2,2025-01-01
1,5.4,0.3,2.6,0.0,0.0,0.0,25.1,2025-01-02
2,3.2,-1.9,0.4,0.0,0.0,0.0,17.1,2025-01-03
3,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1,2025-01-04
4,0.3,-3.6,-2.2,0.0,0.0,0.0,19.9,2025-01-05


In [ ]:
weather_daily.to_csv('../data/citibike/JC/jersey_weather_2025.csv', index = False)

In [ ]:
weather_daily["date"] = pd.to_datetime(weather_daily["date"])

weather_daily.dtypes

temperature_2m_max            float64
temperature_2m_min            float64
temperature_2m_mean           float64
precipitation_sum             float64
rain_sum                      float64
snowfall_sum                  float64
wind_speed_10m_max            float64
date                   datetime64[us]
dtype: object

In [ ]:
fig = px.line(
    weather_daily,
    x="date",
    y="temperature_2m_mean",
    title="Daily Average Temperature Over Time",
    markers=False
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Average Temperature",
    hovermode="x unified"
)

fig.show()

In [ ]:
temperature_long = weather_daily.melt(
    id_vars="date",
    value_vars=[
        "temperature_2m_max",
        "temperature_2m_min",
        "temperature_2m_mean"
    ],
    var_name="temperature_type",
    value_name="temperature"
)

temperature_long.head()

,date,temperature_type,temperature
0,2025-01-01,temperature_2m_max,10.9
1,2025-01-02,temperature_2m_max,5.4
2,2025-01-03,temperature_2m_max,3.2
3,2025-01-04,temperature_2m_max,-0.1
4,2025-01-05,temperature_2m_max,0.3


## Homework part1


In [ ]:
temperature_long['temperature_type'] = (
    temperature_long['temperature_type']
    .str.replace('temperature_', '', regex=False)

)
temperature_long.head()

,date,temperature_type,temperature
0,2025-01-01,2m_max,10.9
1,2025-01-02,2m_max,5.4
2,2025-01-03,2m_max,3.2
3,2025-01-04,2m_max,-0.1
4,2025-01-05,2m_max,0.3


## Homework 


### Wind Speed

In [ ]:
fig = px.line(
    weather_daily,
    x="date",
    y="wind_speed_10m_max",
    title="Daily wind Speed Over Time",
    markers=False
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="wind_speed_10m_max",
    hovermode="x unified"
)

fig.show()

## Homework part 1

### Precipitation

In [ ]:
fig = px.line(
   weather_daily,
    x="date",
    y="precipitation_sum",
   
    title="Daily Precipitation ",
    markers=False
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="precipitation_sum",
    hovermode="x unified"
)

fig.show()

In [ ]:
fig = px.line(
    temperature_long,
    x="date",
    y="temperature",
    color="temperature_type",
    title="Daily Temperature: Maximum, Minimum, and Average",
    markers=False
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Temperature",
    hovermode="x unified"
)

fig.show()

## Homework Part 2

### Day_of_week

In [40]:
citibike_df.info()

<class 'pandas.DataFrame'>
Index: 47864 entries, 0 to 48473
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   ride_id                47864 non-null  str           
 1   rideable_type          47864 non-null  str           
 2   started_at             47864 non-null  datetime64[us]
 3   ended_at               47864 non-null  datetime64[us]
 4   start_station_name     47864 non-null  str           
 5   start_station_id       47864 non-null  str           
 6   end_station_name       47864 non-null  str           
 7   end_station_id         47864 non-null  str           
 8   start_lat              47864 non-null  float64       
 9   start_lng              47864 non-null  float64       
 10  end_lat                47864 non-null  float64       
 11  end_lng                47864 non-null  float64       
 12  member_casual          47864 non-null  str           
 13  ride_duration_min

In [41]:
day_of_week = (citibike_df
                 .groupby('day_of_week',as_index=False)
                 .agg(number_of_rides = ('ride_id',"count"))
                 )
day_of_week

,day_of_week,number_of_rides
0,Friday,6386
1,Monday,8225
2,Saturday,5917
3,Sunday,4090
4,Thursday,7200
5,Tuesday,7520
6,Wednesday,8526


In [42]:
fig = px.bar(
    day_of_week,
    x="day_of_week",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Day of Week"
)

fig.update_layout(
    xaxis_title="day_of_week",
    yaxis_title="Number of Rides",
)

fig.show()

### Seasons

In [45]:
season = (citibike_df
                 .groupby('season',as_index=False)
                 .agg(number_of_rides = ('ride_id',"count"))
                 )
season

,season,number_of_rides
0,Autumn,2
1,Winter,47862


In [46]:
fig = px.bar(
    season,
    x="season",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Day of Week"
)

fig.update_layout(
    xaxis_title="season",
    yaxis_title="Number of Rides",
)

fig.show()

### Hourly

In [49]:
hour = (citibike_df
                 .groupby('hour',as_index=False)
                 .agg(number_of_rides = ('ride_id',"count"))
                 )
hour

,hour,number_of_rides
0,0,565
1,1,356
2,2,208
3,3,121
4,4,242
5,5,624
6,6,1453
7,7,2730
8,8,3436
9,9,2082


In [50]:
fig = px.bar(
    hour,
    x="hour",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Day of Week"
)

fig.update_layout(
    xaxis_title="hour",
    yaxis_title="Number of Rides",
)

fig.show()

## Step 13: Homework

In [ ]:
monthly_rides = (
    citibike_df
    .groupby("month", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

monthly_rides

,month,number_of_rides
0,2024-12,2
1,2025-01,50589
2,2025-02,45250
3,2025-03,73277
4,2025-04,81533
5,2025-05,93202
6,2025-06,96736
7,2025-07,107374
8,2025-08,108001
9,2025-09,115580


In [ ]:
fig = px.bar(
    monthly_rides,
    x="month",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Month"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Rides",
)

fig.show()

In [ ]:
season_rides = (
    citibike_df
    .groupby("season", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

season_order = ["Winter", "Spring", "Summer", "Autumn"]

season_rides["season"] = pd.Categorical(
    season_rides["season"],
    categories=season_order,
    ordered=True
)

season_rides = season_rides.sort_values("season")

season_rides

,season,number_of_rides
3,Winter,143703
1,Spring,248012
2,Summer,312111
0,Autumn,295399


In [ ]:
fig = px.bar(
    season_rides,
    x="season",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Season",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Season",
    yaxis_title="Number of Rides"
)

fig.show()

In [ ]:
top_start_stations = (
    citibike_df
    .dropna(subset=["start_station_name"])
    .groupby("start_station_name", as_index=False)
    .agg(
        number_of_departures=("ride_id", "count")
    )
    .sort_values("number_of_departures", ascending=False)
    .head(10)
)

top_start_stations

,start_station_name,number_of_departures
52,Grove St PATH,45004
58,Hoboken Terminal - Hudson St & Hudson Pl,25889
53,Hamilton Park,22259
95,River St & Newark St,21383
86,Newport PATH,20663
18,Bergen Ave & Sip Ave,20398
44,Exchange Pl,20008
0,11 St & Washington St,19481
94,River St & 1 St,19143
87,Newport Pkwy,18720


In [ ]:
fig = px.bar(
    top_start_stations.sort_values("number_of_departures"),
    x="number_of_departures",
    y="start_station_name",
    orientation="h",
    title="Top 10 Start Stations by Number of Departures",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Departures",
    yaxis_title="Start Station"
)

fig.show()

In [ ]:
top_end_stations = (
    citibike_df
    .dropna(subset=["end_station_name"])
    .groupby("end_station_name", as_index=False)
    .agg(
        number_of_arrivals=("ride_id", "count")
    )
    .sort_values("number_of_arrivals", ascending=False)
    .head(10)
)

top_end_stations

,end_station_name,number_of_arrivals
232,Grove St PATH,47758
241,Hoboken Terminal - Hudson St & Hudson Pl,26639
233,Hamilton Park,22353
347,River St & Newark St,22113
317,Newport PATH,20701
73,Bergen Ave & Sip Ave,20368
207,Exchange Pl,20160
7,11 St & Washington St,19505
318,Newport Pkwy,18707
346,River St & 1 St,18515


In [ ]:
fig = px.bar(
    top_end_stations.sort_values("number_of_arrivals"),
    x="number_of_arrivals",
    y="end_station_name",
    orientation="h",
    title="Top 10 End Stations by Number of Arrivals",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Arrivals",
    yaxis_title="End Station"
)

fig.show()

In [ ]:
daily_rides = (
    citibike_df
    .groupby("date", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)
daily_rides["date"] = pd.to_datetime(daily_rides["date"])
daily_rides.head()

,date,number_of_rides
0,2024-12-31,2
1,2025-01-01,1179
2,2025-01-02,1710
3,2025-01-03,1770
4,2025-01-04,1337


In [ ]:
weather_daily.head()

,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max,date
0,10.9,3.9,7.4,4.5,4.5,0.0,23.2,2025-01-01
1,5.4,0.3,2.6,0.0,0.0,0.0,25.1,2025-01-02
2,3.2,-1.9,0.4,0.0,0.0,0.0,17.1,2025-01-03
3,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1,2025-01-04
4,0.3,-3.6,-2.2,0.0,0.0,0.0,19.9,2025-01-05


In [ ]:
bike_weather_daily = daily_rides.merge(
    weather_daily,
    on="date",
    how="left"
)

bike_weather_daily.head()

,date,number_of_rides,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2024-12-31,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01,1179,10.9,3.9,7.4,4.5,4.5,0.0,23.2
2,2025-01-02,1710,5.4,0.3,2.6,0.0,0.0,0.0,25.1
3,2025-01-03,1770,3.2,-1.9,0.4,0.0,0.0,0.0,17.1
4,2025-01-04,1337,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1


In [ ]:
bike_weather_daily.isna().sum()

date                   0
number_of_rides        0
temperature_2m_max     1
temperature_2m_min     1
temperature_2m_mean    1
precipitation_sum      1
rain_sum               1
snowfall_sum           1
wind_speed_10m_max     1
dtype: int64

In [ ]:
fig = px.scatter(
    bike_weather_daily,
    x="temperature_2m_mean",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Average Temperature"
)

fig.update_layout(
    xaxis_title="Average Daily Temperature",
    yaxis_title="Number of Rides"
)

fig.show()

In [ ]:
fig = px.scatter(
    bike_weather_daily,
    x="wind_speed_10m_max",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Maximum Wind Speed"
)

fig.update_layout(
    xaxis_title="Maximum Wind Speed",
    yaxis_title="Number of Rides"
)

fig.show()

In [ ]:
import plotly.graph_objects as go


fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["number_of_rides"],
        mode="lines",
        name="Daily Rides",
        yaxis="y1"
    )
)

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["temperature_2m_mean"],
        mode="lines",
        name="Daily Average Temperature",
        yaxis="y2"
    )
)

fig.update_layout(
    title="Daily Rides and Daily Average Temperature",
    xaxis=dict(
        title="Day"
    ),
    yaxis=dict(
        title="Daily Rides",
        side="left"
    ),
    yaxis2=dict(
        title="Daily Average Temperature",
        overlaying="y",
        side="right"
    ),
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()

In [ ]:
import geopandas as gpd
from urllib.parse import urlencode
from pathlib import Path


url = '../data/citibike/JC/jersey-city-neighborhoods.geojson'

jersey_city = gpd.read_file(url)

jersey_city.head()

DataSourceError: '../data/citibike/JC/jersey-city-neighborhoods.geojson' not recognized as being in a supported file format. It might help to specify the correct driver explicitly by prefixing the file path with '<DRIVER>:', e.g. 'CSV:path'.

In [ ]:
jersey_city.info()

NameError: name 'jersey_city' is not defined

In [ ]:
jersey_city = jersey_city.to_crs("EPSG:4326")

NameError: name 'jersey_city' is not defined